# CorrDiff - Fase 11 - Análise Formal de Escalas Espaciais

Decomposição Haar, energia espacial, associações predictor-radar por escala e coarse graining.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

OUT = Path('../analysis_outputs/11_spatial_scales')
summary = json.loads((OUT/'analysis_summary.json').read_text())
summary


## 1. Energia espacial Haar

In [ ]:
energy = pd.read_parquet(OUT/'haar_scale_energy.parquet')
radar_energy = energy[(energy.field_group.eq('radar')) & (energy.orientation.eq('ALL_DETAIL'))]
display(radar_energy[['field','nominal_support_km','detail_energy_fraction','rms_coefficient']])


## 2. Perfis de energia dos predictors

In [ ]:
pred_energy = energy[(energy.field_group.eq('predictor')) & (energy.orientation.eq('ALL_DETAIL'))]
for predictor in ['tcwv','r_500','r_850','t2m','t_850','u10','v10']:
    t = pred_energy[pred_energy.field.eq(predictor)].sort_values('nominal_support_km')
    plt.figure(figsize=(7,4))
    plt.plot(t.nominal_support_km, t.detail_energy_fraction, marker='o')
    plt.xscale('log', base=2)
    plt.xlabel('Suporte Haar nominal (km)')
    plt.ylabel('Fração da energia de detalhe')
    plt.title(predictor)
    plt.tight_layout()
    plt.show()


## 3. Associação Haar na mesma escala

In [ ]:
haar = pd.read_parquet(OUT/'haar_same_scale_associations.parquet')
pooled = haar[haar.orientation.eq('ALL_DETAIL')]
for radar_field in ['target_log1p','ge_30','ge_40','ge_45']:
    for predictor in ['tcwv','r_500','r_850','t2m','t_850','wind_speed_10']:
        t = pooled[(pooled.radar_field.eq(radar_field)) & (pooled.predictor.eq(predictor))].sort_values('nominal_support_km')
        plt.figure(figsize=(7,4))
        plt.plot(t.nominal_support_km, t.pearson_r, marker='o')
        plt.xscale('log', base=2)
        plt.axhline(0)
        plt.xlabel('Suporte Haar nominal (km)')
        plt.ylabel('Pearson r')
        plt.title(f'{predictor} × {radar_field}')
        plt.tight_layout()
        plt.show()


## 4. Orientação espacial

In [ ]:
t = haar[(haar.radar_field.eq('ge_45')) & (~haar.orientation.eq('ALL_DETAIL'))]
display(t.sort_values(['nominal_support_km','pearson_r'], ascending=[True,False]))


## 5. Coarse graining raw vs centered

In [ ]:
coarse = pd.read_parquet(OUT/'coarse_grain_associations.parquet')
for predictor in ['tcwv','r_500','r_850','t2m','t_850','wind_speed_10']:
    t = coarse[(coarse.predictor.eq(predictor)) & (coarse.radar_field.eq('ge_40'))]
    for mode, g in t.groupby('association_mode'):
        g = g.sort_values('nominal_block_width_km')
        plt.figure(figsize=(7,4))
        plt.plot(g.nominal_block_width_km, g.pearson_r, marker='o')
        plt.axhline(0)
        plt.xlabel('Bloco nominal (km)')
        plt.ylabel('Pearson r')
        plt.title(f'{predictor} × >=40 dBZ - {mode}')
        plt.tight_layout()
        plt.show()


## 6. Correlação entre energia de escalas diferentes

In [ ]:
cross = pd.read_parquet(OUT/'haar_patch_energy_cross_scale.parquet')
for predictor in ['tcwv','r_500','r_850','t2m','t_850','wind_speed_10']:
    t = cross[(cross.predictor.eq(predictor)) & (cross.radar_field.eq('ge_45'))]
    m = t.pivot(index='predictor_nominal_support_km', columns='radar_nominal_support_km', values='pearson_r')
    display(m)


## 7. Sazonalidade

In [ ]:
season = pd.read_parquet(OUT/'haar_same_scale_associations_by_season.parquet')
t = season[(season.radar_field.eq('ge_45')) & (season.orientation.eq('ALL_DETAIL'))]
display(t.sort_values(['season_code','nominal_support_km','pearson_r'], ascending=[True,True,False]))


## Regra de interpretação

Os suportes Haar e os blocos são escalas nominais na grade CorrDiff. Eles não devem ser chamados de resolução física do ERA5. Priorize padrões que sejam coerentes entre energia, associação Haar e coarse graining.